<a href="https://colab.research.google.com/github/Ruchi74/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ruchi74/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [8]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load the FlyRank starter dataset
data_url = (
    "https://raw.githubusercontent.com/"
    "Ruchi74/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Dataset loaded successfully!
Dataset shape: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [9]:

# STEP 3 — CHECK AVAILABLE TARGET LABELS

print("Dataset columns that may contain labels or outcomes:")
for col in df.columns:
    if any(word in col.lower() for word in ["label", "target", "declin", "trend"]):
        print("-", col)

print("\nTrend direction counts (inspection only):")
print(df["trend_direction"].value_counts(dropna=False))

print("\nTrend percentage summary (inspection only):")
print(df["trend_pct"].describe())

print("\nRecent and previous period columns:")
print([
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d"
])


Dataset columns that may contain labels or outcomes:
- trend_direction
- trend_pct

Trend direction counts (inspection only):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend percentage summary (inspection only):
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Recent and previous period columns:
['impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']


In [10]:

# STEP 4 — DEFINE A TARGET FROM RECENT VS PREVIOUS IMPRESSIONS

# Keep rows where both periods have usable impression counts
model_df = df[
    df["impressions_last_30d"].notna() &
    df["impressions_prev_30d"].notna() &
    (df["impressions_prev_30d"] >= 100)
].copy()

# Target: 1 means impressions declined by at least 20%;
# 0 means they did not decline by that amount.
model_df["target_decline"] = (
    model_df["impressions_last_30d"]
    <= 0.8 * model_df["impressions_prev_30d"]
).astype(int)

print("Rows available for modeling:", len(model_df))
print("\nTarget counts:")
print(model_df["target_decline"].value_counts().sort_index())

print("\nTarget proportions:")
print(
    model_df["target_decline"]
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

print("\nTarget definition:")
print("1 = impressions declined by at least 20%")
print("0 = impressions did not decline by at least 20%")


Rows available for modeling: 18010

Target counts:
target_decline
0     6909
1    11101
Name: count, dtype: int64

Target proportions:
target_decline
0    0.384
1    0.616
Name: proportion, dtype: float64

Target definition:
1 = impressions declined by at least 20%
0 = impressions did not decline by at least 20%


In [19]:


# STEP 5 — CLEAN FEATURE SPLIT

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

excluded_columns = [
    "content_id", "client_id", "target_decline",
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "ctr", "impressions_90d", "clicks_90d",
    "days_with_impressions", "impression_tier"
]

X = model_df.drop(columns=excluded_columns, errors="ignore")
y = model_df["target_decline"]

# Encode categorical columns
X = pd.get_dummies(X, drop_first=True, dtype=int)
X = X.replace([np.inf, -np.inf], np.nan)

# Split before imputing missing values
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Learn imputation values from training data only
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", X_train.shape[1])
print("Missing values in training:", X_train.isna().sum().sum())
print("Missing values in testing:", X_test.isna().sum().sum())


Training rows: 14408
Testing rows: 3602
Features: 47
Missing values in training: 0
Missing values in testing: 0


In [20]:


# STEP 6 — CHECK FOR TARGET LEAKAGE

print("Target definition:")
print("Decline = last 30-day impressions <= 80% of previous 30-day impressions")

print("\nExcluded target-related columns:")
print([
    "impressions_last_30d",
    "impressions_prev_30d",
    "trend_direction",
    "trend_pct"
])

print("\nAre target-related columns in model features?")
for col in [
    "impressions_last_30d",
    "impressions_prev_30d",
    "trend_direction",
    "trend_pct"
]:
    print(f"{col}: {col in X_train.columns}")

print("\nLeakage check complete.")



Target definition:
Decline = last 30-day impressions <= 80% of previous 30-day impressions

Excluded target-related columns:
['impressions_last_30d', 'impressions_prev_30d', 'trend_direction', 'trend_pct']

Are target-related columns in model features?
impressions_last_30d: False
impressions_prev_30d: False
trend_direction: False
trend_pct: False

Leakage check complete.


In [21]:


# STEP 7 — RETRAIN THE DECISION TREE

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("UPDATED DECISION TREE RESULTS")
print("-----------------------------")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


UPDATED DECISION TREE RESULTS
-----------------------------
Accuracy: 0.669
Precision: 0.743
Recall: 0.707
F1 Score: 0.725

Confusion Matrix:
[[ 840  542]
 [ 650 1570]]


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Feature Importance Interpretation

The Decision Tree model identified `content_age_days` as the most important feature, followed by `days_with_sessions`, `scroll_rate`, and `avg_position`. These features contributed most to the model's decisions. Some features had zero importance because the tree did not use them to make splits.

Feature importance indicates predictive contribution, not causation. The model achieved an F1 score of 0.725 and accuracy of 66.9% on the test set. Since the target was created from historical impression changes, these results should be treated as a retrospective proxy exercise rather than evidence of reliable future prediction.

### Model vs. Week 4 Baseline

The Decision Tree achieved an accuracy of 0.669 and an F1 score of 0.725. The Week 4 baseline achieved an accuracy of 0.526 and an F1 score of 0.581.

The Decision Tree performed better on the evaluated test set across accuracy, precision, recall, and F1 score. Its F1 score was 0.144 higher than the baseline.

However, the Week 4 baseline was originally designed to identify low-CTR pages rather than predict impression declines. Therefore, this comparison is preliminary and should not be interpreted as a fully equivalent comparison. The target is based on historical impression changes, so future predictive performance has not been established.

In [22]:




# STEP 8 — UPDATED BASELINE COMPARISON

baseline_pred = (
    (model_df.loc[X_test.index, "impressions_90d"] >= 500) &
    (model_df.loc[X_test.index, "avg_position"] > 0) &
    (model_df.loc[X_test.index, "avg_position"] <= 20) &
    (model_df.loc[X_test.index, "ctr"] < 0.5)
).astype(int)

comparison = pd.DataFrame({
    "Method": ["Week 4 Baseline", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, y_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, y_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    ]
})

display(comparison.round(3))


,Method,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,0.526,0.639,0.533,0.581
1,Decision Tree,0.669,0.743,0.707,0.725


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.